In [14]:
from IPython.display import display

from eki_dev.aws_service import AwsService
from aws_cluster import pest_cluster
from aws_cluster.cluster_utils import (check_resource_creation_status,
check_stack_creation_status)



# EKI PEST_HP Cluster Dashboard

## Path to Yaml Configuration

In [15]:
#path_yaml_config = 's3://scratch-marco/parameters_no_elb.yaml'
path_yaml_config = 's3://eki-wr-proj-c20037/eki-pest-cluster-config/pest_cluster_jacobian_maneta.yaml'

## STEP 1: Create Cluster

In [16]:
if pest_cluster.create_pest_cluster_stack(path_yaml_config, cf_template="https://eki-cf-templates.s3.us-west-1.amazonaws.com/PestCluster-cf-no_elb-tpl.yaml"):
    print('Pest cluster stack creation initiated')

Pest cluster stack creation initiated


## Monitor Cluster Creation and Status

Use the cell below to monitor the status of the different elastic components that form the cluster. Initially the service is created with a single agent waiting for work from the main task that holds the host process (created in Step 2). 

In [22]:

cluster_status = check_stack_creation_status("PestClusterInfrastructure")
print(f"The current status of the cluster formation is {cluster_status}")
print("Do not proceed until the cluster formation is complete.")
display(check_resource_creation_status("PestClusterInfrastructure"))

The current status of the cluster formation is CREATE_COMPLETE
Do not proceed until the cluster formation is complete.


,LogicalResourceId,ResourceStatus
0,ECSService,CREATE_COMPLETE
1,ECSTaskDefinition00taskdefinitionpestagent600ELdqq,CREATE_COMPLETE
2,EIP,CREATE_COMPLETE
3,InstanceSGrpPestHPIncoming,CREATE_COMPLETE
4,MainInstance,CREATE_COMPLETE
5,NatGateway,CREATE_COMPLETE
6,Route,CREATE_COMPLETE


## STEP 2: Create main task

Creates the main task in the cluster running the PEST host process. The agents in the cluster will start running model instances as soon as the task is created. 

In [21]:
pest_cluster.add_docker_context_to_main_instance(fn_config_yaml=path_yaml_config)

Public IP: 18.144.68.61
creating docker context with name pst_zone7_noptmax_minus2
Creating docker context for ssh://ubuntu@18.144.68.61:22
To initiate the main task and interact with the PEST HP Cluster open a terminal and:
1. $> edamame generate-makefile --image-name zone7 --repo-name zone7
2. $> docker context use pst_zone7_noptmax_minus2
3. $> make TAG=dev pull_aws
4. $> docker run -d -v /home/ubuntu/efs:/home/eki/efs -p 4004:4004 zone7:dev pest_hp pst_zone7_noptmax_minus2 /h :4004


In [18]:
# if check_stack_creation_status("PestClusterInfrastructure") == 'CREATE_COMPLETE':
#     res = pest_cluster.create_main_task(path_yaml_config)

Waiting for Main Task to be Created...
Main Task Created. Registering IP in Target Group
Main Instance IP: 10.10.27.65


In [20]:
desired_number_agents = 32


### Do not modify anything below this line ######
pest_cluster.update_number_agents(path_yaml_config, desired_number_agents);


In [22]:
# cf = AwsService.from_service('cloudformation')
# stack_resources = cf.client.describe_stack_resources(StackName="PestClusterInfrastructure")
# for resource in stack_resources['StackResources']:
#     if resource['ResourceType'] == "AWS::ECS::Service":
#         service_arn = resource['PhysicalResourceId']
#         print(service_arn)
        

arn:aws:ecs:us-west-1:054507568115:service/PestCluster/agent_private_net


In [5]:
pest_cluster.list_agent_tasks(path_yaml_config)


,0
0,arn:aws:ecs:us-west-1:054507568115:task/PestCl...


In [ ]:
logs_client = AwsService.from_service("logs").client
 
response = logs_client.get_log_events(
        logGroupName="/ecs/pest_agent",
        logStreamName="ecs/model/f7389a4dc2374efb91e3a2be853b92a8",
        limit=100,
        startFromHead=False
)
for event in response["events"]:
        print(event["timestamp"], event["message"])

In [16]:
# ecs = AwsService.from_service('ecs')
# service_arns = ecs.client.list_services(cluster="PestCluster")["serviceArns"]
# print(service_arns)
# ecs.client.list_tasks(
#         cluster="PestCluster",
#         serviceName=service_arns[0]
#     )

#ecs.client.describe_task_sets(cluster="PestCluster", service=service_arn)

In [12]:
main_task = cf.client.describe_tasks(cluster="PestCluster", tasks=['arn:aws:ecs:us-west-1:054507568115:task/PestCluster/f65a622afc4e434dbfdaef14d7a550bb'])


In [18]:
main_task

{'tasks': [{'attachments': [],
   'attributes': [{'name': 'ecs.cpu-architecture', 'value': 'x86_64'}],
   'availabilityZone': 'us-west-1c',
   'capacityProviderName': 'Infra-ECS-Cluster-PestCluster-a8643df8-EC2CapacityProvider-HxlrJRzZVZ1l',
   'clusterArn': 'arn:aws:ecs:us-west-1:054507568115:cluster/PestCluster',
   'connectivity': 'CONNECTED',
   'connectivityAt': datetime.datetime(2024, 10, 13, 18, 10, 16, 300000, tzinfo=tzlocal()),
   'containerInstanceArn': 'arn:aws:ecs:us-west-1:054507568115:container-instance/PestCluster/45479f3eeec740959e36a01f125834d1',
   'containers': [{'containerArn': 'arn:aws:ecs:us-west-1:054507568115:container/PestCluster/f65a622afc4e434dbfdaef14d7a550bb/13ec9e27-8ac3-4449-baf4-428dd4a682e5',
     'taskArn': 'arn:aws:ecs:us-west-1:054507568115:task/PestCluster/f65a622afc4e434dbfdaef14d7a550bb',
     'name': 'model',
     'image': '054507568115.dkr.ecr.us-west-1.amazonaws.com/ww_2024:dev',
     'imageDigest': 'sha256:0198069cc2cb5d2b7eb95c461ffcdbbc81213

In [16]:
main_task['tasks'][0]['containerInstanceArn']

'arn:aws:ecs:us-west-1:054507568115:container-instance/PestCluster/45479f3eeec740959e36a01f125834d1'

In [20]:
main_container = cf.client.describe_container_instances(cluster="PestCluster", 
                                       containerInstances=[main_task['tasks'][0]['containerInstanceArn']])

In [23]:
main_instance = main_container['containerInstances'][0]['ec2InstanceId']

In [22]:
cf = AwsService.from_service('ec2')

In [26]:
main_instance_ip = cf.client.describe_instances(InstanceIds = [main_instance])

In [32]:
main_instance_ip = main_instance_ip['Reservations'][0]['Instances'][0]['PrivateIpAddress']

In [36]:
cf = AwsService.from_service('ecs')

'arn:aws:elasticloadbalancing:us-west-1:054507568115:targetgroup/PestMain/989eb9bf9ee12230'

In [46]:
cf = AwsService.from_service('elbv2')
cf.client.register_targets(TargetGroupArn=target_group_arn,
                           Targets=[{
                               'Id': main_instance_ip,
                               'Port': 4004,
                           }])

{'ResponseMetadata': {'RequestId': '8633782f-f43d-4851-b8cd-832c476333e3',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '8633782f-f43d-4851-b8cd-832c476333e3',
   'content-type': 'text/xml',
   'content-length': '253',
   'date': 'Mon, 14 Oct 2024 02:19:12 GMT'},
  'RetryAttempts': 0}}

In [23]:
import pandas as pd
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'matplotlib'

In [76]:
df = pd.DataFrame({'A': [1, 2], 'B': [3, 4]})
df

,A,B
0,1,3
1,2,4


In [82]:
    
df_style = df.style.apply(lambda x: ['background-color: green'], axis=0)
    

In [84]:
df_style

,A,B
0,1,3
1,2,4


In [19]:
pest_cluster.terminate_cluster()

ServiceNotActiveException: An error occurred (ServiceNotActiveException) when calling the UpdateService operation: Service was not ACTIVE.